In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot+R2plus1D 6 Layer"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

/tmp/ipykernel_3929179/4234141958.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2025-02-12 14:42:57,874]::2549926179::DEBUG::=============fullshot+R2plus1D 6 Layer+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
# test_set_data=getdata("/home/ssrlab/qx/code/test/video-swin-transformer-pytorch/data/15Frames/test_set.dat","test")
# test_set_data=test_set_data[:datasize("test",batchsz)]
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
# validation_set_data=getdata("/home/ssrlab/qx/video-swin-transformer-pytorch/data/15Frames/margin011/valid_set.dat","valid")

# validation_set_data=reshape_to_expected_input(validation_set_data)
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
# train_set_data=getdata("/home/ssrlab/qx/video-swin-transformer-pytorch/data/15Frames/margin011/train_set.dat","train")

# train_set_data=reshape_to_expected_input(train_set_data)
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [11]:
model=ViViTBackbone(2, 14, 14, 2, 2, 2, 5, 96, 6, 8, 8, model=3)#num_frames=frame_nums,batchsz = batchsz
model.to(device)
criterion = torch.nn.L1Loss().to(device)  # 손실함수
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # 옵티마이저

if test_flage==True:
    checkpoint=torch.load(save_model_file_path.format('model',start_epoch,'pth'), map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    # criterion.load_state_dict(checkpoint["loss"])
train_avg_loss0=[]
val_avg_loss0=[]
print(1)

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131
1


In [12]:
from torchinfo import summary

summary(model, input_size = ((4, 3,15, 224, 224),(4,24,1319,1)), col_names = ['input_size','output_size','num_params'])

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #
ViViTBackbone                                                [4, 3, 15, 224, 224]      [4, 5]                    3,146,016
├─ASTModel: 1-1                                              [4, 1319, 24]             [4, 768]                  885,120
│    └─DistilledVisionTransformer: 2-1                       --                        --                        1,641,680
│    │    └─PatchEmbed: 3-1                                  [4, 1, 24, 1319]          [4, 131, 768]             197,376
│    │    └─Dropout: 3-2                                     [4, 133, 768]             [4, 133, 768]             --
│    │    └─ModuleList: 3-3                                  --                        --                        85,054,464
│    │    └─LayerNorm: 3-4                                   [4, 133, 768]             [4, 133, 768]             1,536
│    └─Sequential: 2-2          

In [13]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [14]:
def imagecrop(imagedata,batchsz):
    # convert_tensor= transforms.ToTensor()
    output=[]
    imagedata=imagedata.permute(0,2,1,3,4)
    for j in range(batchsz):#비디오 데이터 batchsz(9) 개 있다
        firstoutput=[]
        for i in range(len(imagedata[j])):#imagedata에 15개의 프레임이 있다
            # temp = convert_tensor(imagedata[i])
            #이미지를 나누
            temp = imagedata[j][i].unfold(1,2,2).unfold(2,2,2)
            temp = rearrange(temp, 'c h1 w1 h w -> (h1 w1) c h w')
            firstoutput.append(temp)
        firstoutput = torch.stack(firstoutput,0)
        output.append(firstoutput)
    output = torch.stack(output,0)
    return output

In [ ]:
with torch.cuda.device(1):
    max_value = float('inf')
    for i in range(start_epoch, epochs):
        train_avg_loss = 0
        val_avg_loss = 0
        train_avg_o_loss = 0
        train_avg_c_loss = 0
        train_avg_e_loss = 0
        train_avg_a_loss = 0
        train_avg_n_loss = 0
        val_avg_o_loss = 0
        val_avg_c_loss = 0
        val_avg_e_loss = 0
        val_avg_a_loss = 0
        val_avg_n_loss = 0
        for audiodata,fullshot,big_five_data in tqdm(train_dataloader):
            fullshot=fullshot.permute(0,4,1,2,3)
            big_five_data=big_five_data.permute(0,2,1)
            big_five_data=big_five_data.squeeze()
            # fullshot=imagecrop(fullshot,batchsz)
            audiodata = audiodata.to(device)
            fullshot = fullshot.to(device)
            big_five_data = big_five_data.to(device)
            optimizer.zero_grad()  # 기울기가 0이 됩니다.
            hypothesis = model(fullshot,audiodata) # 모델의 예측 결과를 저장합니다.
            loss = criterion(hypothesis, big_five_data)  # 예측된 결과와 실제 태그 사이의 손실 값을 저장합니다.
            loss.backward()  # 역방향 전파입니다. 
            optimizer.step()  
            train_avg_loss += loss  
        train_avg_loss=train_avg_loss/len(train_dataloader)
        with torch.no_grad():#validate
            for audiodata,fullshot,big_five_data in tqdm(val_dataloader):
                fullshot=fullshot.permute(0,4,1,2,3)
                audiodata = audiodata.to(device)
                # fullshot=imagecrop(fullshot,batchsz)
                big_five_data=big_five_data.permute(0,2,1)
                big_five_data=big_five_data.squeeze()
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)
                hypothesis = model(fullshot,audiodata)
                val_loss = criterion(hypothesis, big_five_data)
                hypothesiso,hypothesisc,hypothesise,hypothesisa,hypothesisn=extract_ocean(hypothesis,len(fullshot))
                big_five_datao,big_five_datac,big_five_datae,big_five_dataa,big_five_datan=extract_ocean(big_five_data,len(fullshot))
                val_o_loss = criterion(hypothesiso, big_five_datao) 
                val_c_loss = criterion(hypothesisc, big_five_datac) 
                val_e_loss = criterion(hypothesise, big_five_datae) 
                val_a_loss = criterion(hypothesisa, big_five_dataa) 
                val_n_loss = criterion(hypothesisn, big_five_datan)  
                val_avg_loss += val_loss
                val_avg_o_loss += val_o_loss
                val_avg_c_loss += val_c_loss
                val_avg_e_loss += val_e_loss
                val_avg_a_loss += val_a_loss
                val_avg_n_loss += val_n_loss
            val_avg_loss=val_avg_loss/len(val_dataloader)
            val_avg_o_loss=val_avg_o_loss/len(val_dataloader)
            val_avg_c_loss=val_avg_c_loss/len(val_dataloader)
            val_avg_e_loss=val_avg_e_loss/len(val_dataloader)
            val_avg_a_loss=val_avg_a_loss/len(val_dataloader)
            val_avg_n_loss=val_avg_n_loss/len(val_dataloader)
        torch.cuda.empty_cache()
        # if (1-val_avg_loss)>=(1-max_value):
        start_epoch+=1
        if val_avg_loss < max_value:
            max_value=val_avg_loss
            torch.save({
                'epoch': i+1,
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'loss': val_avg_loss,
            }, save_model_file_path.format('model',start_epoch,'pth'))
        logger.debug('Epoch: {} , 1-MAE: {:.4f} , 1-Training Loss: {:.4f} , 1-val_avg_o_loss:{:.4f} , 1-val_avg_c_loss:{:.4f} , 1-val_avg_e_loss:{:.4f} , 1-val_avg_a_loss:{:.4f} , 1-val_avg_n_loss:{:.4f}'.format(i+1, 1-val_avg_loss, 1-train_avg_loss,1-val_avg_o_loss, 1-val_avg_c_loss, 1-val_avg_e_loss, 1-val_avg_a_loss,1-val_avg_n_loss))
    

  0%|          | 0/1498 [00:00<?, ?it/s]

100%|█████████▉| 1497/1498 [06:26<00:00,  3.90it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)
100%|██████████| 498/498 [00:51<00:00,  9.60it/s]
[2025-02-12 14:50:28,382]::3197456309::DEBUG::Epoch: 1 , 1-MAE: 0.8933 , 1-Training Loss: 0.8860 , 1-val_avg_o_loss:0.8942 , 1-val_avg_c_loss:0.8991 , 1-val_avg_e_loss:0.8738 , 1-val_avg_a_loss:0.8973 , 1-val_avg_n_loss:0.9023
100%|██████████| 498/498 [00:51<00:00,  9.69it/s]
[2025-02-12 14:57:49,235]::3197456309::DEBUG::Epoch: 2 , 1-MAE: 0.9065 , 1-Training Loss: 0.9030 , 1-val_avg_o_loss:0.8944 , 1-val_avg_c_loss:0.9065 , 1-val_avg_e_loss:0.9133 , 1-val_avg_a_loss:0.9070 , 1-val_avg_n_loss:0.9112
100%|██████████| 498/498 [00:52<00:0

In [ ]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))